In [68]:
import pandas as pd
import numpy as np

# Load crime data
crime = pd.read_excel('csv_files/2022 Offenses by County and City Township.xlsx', header=2)

# Load Census files
income       = pd.read_csv('csv_files/Median Household Information.csv', skiprows=1)
vacancy      = pd.read_csv('csv_files/Vacancy Status.csv')
owned_rented = pd.read_csv('csv_files/Owned vs Rented.csv')
home_value   = pd.read_csv('csv_files/Median Home Value.csv')
population   = pd.read_csv('csv_files/Total Population.csv')

print("All files loaded successfully")

All files loaded successfully


In [69]:
# Cleaning the crime data
crime.head()

# Keep only columns we need from crime data
crime_clean = crime[['Unnamed: 0', 22001, 22002]].copy()

# Create total burglary column
crime_clean['burglary_total'] = crime_clean[22001] + crime_clean[22002]

# Keep only place and total
crime_clean = crime_clean[['Unnamed: 0', 'burglary_total']]
crime_clean.columns = ['place', 'burglary_total']

# Clean up place name - strip whitespace
crime_clean['place'] = crime_clean['place'].str.strip()

print(crime_clean.shape)
print(crime.shape)
crime_clean.head()

(1927, 2)
(1927, 28)


,place,burglary_total
0,Alcona,5
1,Alcona Township,0
2,Caledonia Township,0
3,Curtis Township,3
4,Greenbush Township,0


In [70]:
# Cleaning and combining the predicting variable data

def clean_place_name(name):
    name = str(name).replace(', Michigan', '')
    name = name.replace(' city', '').replace(' village', '')
    name = name.replace(' township', '').replace(' CDP', '')
    return name.strip()

def get_transposed_row(df, row_label):
    est_cols = [c for c in df.columns if ', Michigan' in c and '!!Margin' not in c]
    row = df[df['Label (Grouping)'].str.strip() == row_label]
    result = row[est_cols].T.reset_index()
    result.columns = ['place', 'value']
    result['place'] = result['place'].str.replace('!!Estimate', '').str.strip().apply(clean_place_name)
    return result
    
income_clean = income[['Geographic Area Name', 
                        'Estimate!!Households!!Median income (dollars)']].copy()
income_clean.columns = ['place', 'median_income']
income_clean['place'] = income_clean['place'].apply(clean_place_name)
income_clean = income_clean[income_clean['place'] != 'Michigan']

vacancy_clean = get_transposed_row(vacancy2, 'Total:')
vacancy_clean.columns = ['place', 'vacant_units']

owned_clean = get_transposed_row(owned_rented2, 'Owner occupied')
owned_clean.columns = ['place', 'owner_occupied']

rented_clean = get_transposed_row(owned_rented2, 'Renter occupied')
rented_clean.columns = ['place', 'renter_occupied']

home_value_clean = get_transposed_row(home_value2, 'Median value (dollars)')
home_value_clean.columns = ['place', 'median_home_value']

population_clean = get_transposed_row(population2, 'Total')
population_clean.columns = ['place', 'population']


census = income_clean.copy()
census = census.merge(vacancy_clean,    on='place', how='inner')
census = census.merge(owned_clean,      on='place', how='inner')
census = census.merge(rented_clean,     on='place', how='inner')
census = census.merge(home_value_clean, on='place', how='inner')
census = census.merge(population_clean, on='place', how='inner')

print("Crime shape:", crime_clean.shape)
print("Census shape:", census.shape)
census.head()

Crime shape: (1927, 2)
Census shape: (869, 7)


,place,median_income,vacant_units,owner_occupied,renter_occupied,median_home_value,population
0,Addison,54519,19,160,82,"132,700",585
1,Adrian,50209,"1,078","4,414","3,890","119,900","20,493"
2,Advance,110417,106,181,14,"549,600",372
3,Ahmeek,41250,55,49,4,"75,000",90
4,Akron,50179,22,115,25,"75,500",312


In [71]:
# Getting rid of places besides cities
county_indicators = ['Township', 'township', 'city', 'City', 'village',
                     'Village', 'CDP', 'Heights', 'Park', 'Falls',
                     'Rapids', 'Beach', 'Harbor', 'Hills', 'Correctional']

has_indicator  = crime_clean['place'].str.contains('|'.join(county_indicators), na=False)

# Keep rows with an indicator OR rows that match a census place name
in_census      = crime_clean['place'].isin(census['place'])
crime_filtered = crime_clean[has_indicator | in_census].copy()

print("Crime after filtering:", crime_filtered.shape)

# Now merge
df = crime_filtered.merge(census, on='place', how='inner')

# Fix numeric columns that have commas - convert to actual numbers
for col in ['vacant_units', 'owner_occupied', 'renter_occupied', 'median_home_value', 'population']:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '', regex=False), errors='coerce')

# Also make sure median_income is numeric
df['median_income'] = pd.to_numeric(df['median_income'], errors='coerce')

# Fix duplicate places issue
df = df.groupby('place', as_index=False).agg({
    'burglary_total':    'sum',
    'median_income':     'first',
    'vacant_units':      'first',
    'owner_occupied':    'first',
    'renter_occupied':   'first',
    'median_home_value': 'first',
    'population':        'first'
})

print("Shape after deduplication:", df.shape)
print("Any duplicates left:", df.duplicated('place').any())
df.head()

Crime after filtering: (1816, 2)
Shape after deduplication: (513, 8)
Any duplicates left: False


,place,burglary_total,median_income,vacant_units,owner_occupied,renter_occupied,median_home_value,population
0,Addison,3,54519.0,19,160,82,132700.0,585
1,Adrian,54,50209.0,1078,4414,3890,119900.0,20493
2,Ahmeek,0,41250.0,55,49,4,75000.0,90
3,Akron,1,50179.0,22,115,25,75500.0,312
4,Alanson,0,65139.0,118,328,74,121200.0,943


In [72]:
# Calculate burglary rate per 1,000 residents
df['burglary_rate'] = (df['burglary_total'] / df['population']) * 1000

# Create high/low risk label based on state median burglary rate
median_rate = df['burglary_rate'].median()
print("State median burglary rate:", round(median_rate, 2))

df['high_risk'] = (df['burglary_rate'] > median_rate).astype(int)

print("Label distribution:")
print(df['high_risk'].value_counts())
df.head()

State median burglary rate: 1.12
Label distribution:
high_risk
0    257
1    256
Name: count, dtype: int64


,place,burglary_total,median_income,vacant_units,owner_occupied,renter_occupied,median_home_value,population,burglary_rate,high_risk
0,Addison,3,54519.0,19,160,82,132700.0,585,5.128205,1
1,Adrian,54,50209.0,1078,4414,3890,119900.0,20493,2.635046,1
2,Ahmeek,0,41250.0,55,49,4,75000.0,90,0.000000,0
3,Akron,1,50179.0,22,115,25,75500.0,312,3.205128,1
4,Alanson,0,65139.0,118,328,74,121200.0,943,0.000000,0
